# Assessment 1 — Weeks 1–11

**Time:** 6 hours, open book (your notes, the course notebooks, library documentation) — but your own work: no help from other people or AI assistants.
**Marks:** 100 in eight tasks; the points are given with each task. **Pass:** 60.
**Rules:** every function must run from a fresh kernel in the course environment (`requirements.txt`). Written answers count as much as code: keep them short, precise and in your own words. The test cells check that your results are **consistent** — they do not tell you whether your numbers are right.

**Submit** the notebook, run from top to bottom, for marking. The answer key in `solutions/` is opened only after marking.

| Task | Topic | Module | Points |
|---|---|---|---|
| 1 | Three-phase power and power-factor correction | EE foundations | 8 |
| 2 | Newton's method on a power-flow equation | Maths foundations | 8 |
| 3 | 15-minute data across both daylight-saving changes | M1 | 12 |
| 4 | Market clearing as a welfare LP; a carbon switching price | M2 | 12 |
| 5 | A dispatch LP with a ramp limit: prices, KKT, PyPSA | M3 | 16 |
| 6 | A reservoir and its water value | M3 | 12 |
| 7 | Code review: a forecasting function with leaks | Forecasting | 18 |
| 8 | A newsvendor decision and a brief | Forecasting | 14 |

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

# the forecasting module's data generator, found from the repository root (works from assessments/ and assessments/solutions/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "COURSE_PLAN.md").exists())
sys.path.insert(0, str(ROOT / "06_energy_forecasting"))
from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)


def mae(forecast, actual):
    return float(np.abs(np.asarray(forecast, float) - np.asarray(actual, float)).mean())

## Task 1 — Three-phase power (8 points)

A factory draws **200 kW** at a power factor of **0.90 lagging** from a **400 V** (line-to-line) three-phase supply.

(a) Write `three_phase(P_kW, pf, V_LL)` returning the apparent power S (kVA), the reactive power Q (kvar) and the line current I (A).
(b) Write `capacitor_kvar(P_kW, pf_from, pf_to)`: the capacitor bank (kvar) that raises the power factor from `pf_from` to `pf_to` at constant real power. Compute it for 0.90 → 0.98 and the line current afterwards.
(c) In two sentences: why does the network operator care about the factory's power factor, if the energy (kWh) it buys is unchanged?

In [ ]:
def three_phase(P_kW: float, pf: float, V_LL: float) -> dict:
    """Apparent power (kVA), reactive power (kvar) and line current (A) of a balanced three-phase load."""
    ...  # YOUR CODE HERE


def capacitor_kvar(P_kW: float, pf_from: float, pf_to: float) -> float:
    """Capacitor (kvar) that raises the power factor from pf_from to pf_to at constant real power."""
    ...  # YOUR CODE HERE

In [ ]:
before = three_phase(200, 0.90, 400)
after = three_phase(200, 0.98, 400)
qc = capacitor_kvar(200, 0.90, 0.98)
assert np.isclose(before["S_kVA"] ** 2, 200 ** 2 + before["Q_kvar"] ** 2), "the power triangle must close"
assert np.isclose(np.sqrt(3) * 400 * before["I_A"] / 1e3, before["S_kVA"]), "S = sqrt(3) V I"
assert np.isclose(before["Q_kvar"] - qc, after["Q_kvar"]), "the capacitor supplies the difference in Q"
print({k: round(v, 1) for k, v in before.items()}, "| capacitor", round(qc, 1), "kvar | current after", round(after["I_A"], 1), "A")

*Your answer here.*

## Task 2 — Newton's method on a power-flow equation (8 points)

Two buses at 1.0 p.u. voltage are joined by a lossless line of susceptance B = 10 p.u. The power sent across it is P = V₁ V₂ B sin δ, where δ is the angle difference.

(a) Write `newton_angle(P, V1=1.0, V2=1.0, B=10.0, tol=1e-10)` that solves for δ (radians) with Newton's method from δ = 0, returning δ and the number of iterations. It must raise a `ValueError` if no solution exists.
(b) Solve for P = 4 p.u. How many iterations does it need?
(c) In two sentences: why can no angle carry P = 12 p.u., and what does that mean physically?

In [ ]:
def newton_angle(P: float, V1: float = 1.0, V2: float = 1.0, B: float = 10.0, tol: float = 1e-10) -> tuple[float, int]:
    """Angle difference (rad) that carries P across the line, by Newton's method from 0, and the iterations used."""
    ...  # YOUR CODE HERE

In [ ]:
delta, iterations = newton_angle(4.0)
assert abs(10 * np.sin(delta) - 4.0) < 1e-9, "the angle must carry the power"
assert iterations <= 10, "Newton converges in a handful of iterations here"
try:
    newton_angle(12.0)
    raise AssertionError("P = 12 should raise ValueError")
except ValueError:
    pass
print(f"delta = {np.degrees(delta):.3f} degrees after {iterations} iterations")

*Your answer here.*

## Task 3 — 15-minute data across the clock changes (12 points)

`QUARTER_HOURS` (below) holds a site's 15-minute average power in MW, in local time (Europe/Zagreb), for two windows that contain the spring and the autumn clock change of 2025. Three quarter-hours are missing.

(a) Write `clean(series)` returning the series with gaps filled by time interpolation, and the number of quarter-hours that were missing.
(b) Write `hourly_energy(series)` returning energy per hour in MWh.
(c) Write `daily_summary(series)` returning, per local calendar day, the number of hours and the energy in MWh.
(d) In two or three sentences: how many hours do 30 March and 26 October 2025 have, and what goes wrong if the timestamps are first converted to tz-naive local time?

In [ ]:
idx = pd.date_range("2025-03-29", "2025-04-01", freq="15min", tz="Europe/Zagreb", inclusive="left").append(
    pd.date_range("2025-10-25", "2025-10-28", freq="15min", tz="Europe/Zagreb", inclusive="left"))
local_hour = idx.hour.to_numpy() + idx.minute.to_numpy() / 60
QUARTER_HOURS = pd.Series(2000 + 300 * np.sin(np.pi * (local_hour - 6) / 12).clip(0), index=idx, name="MW")
QUARTER_HOURS = QUARTER_HOURS.drop(QUARTER_HOURS.index[[40, 41, 300]])        # three quarter-hours lost in transmission

In [ ]:
def clean(series: pd.Series) -> tuple[pd.Series, int]:
    """Gap-free 15-minute series (time interpolation) and the number of missing quarter-hours."""
    ...  # YOUR CODE HERE


def hourly_energy(series: pd.Series) -> pd.Series:
    """Energy per hour (MWh) from 15-minute average power (MW)."""
    ...  # YOUR CODE HERE


def daily_summary(series: pd.Series) -> pd.DataFrame:
    """Hours and energy (MWh) per local calendar day."""
    ...  # YOUR CODE HERE

In [ ]:
filled, n_missing = clean(QUARTER_HOURS)
assert filled.isna().sum() == 0 and str(filled.index.tz) == "Europe/Zagreb"
energy = hourly_energy(filled)
assert np.isclose((filled * 0.25).sum(), energy.sum()), "resampling must conserve energy"
summary = daily_summary(filled)
assert np.isclose(summary.MWh.sum(), energy.sum())
print("missing quarter-hours:", n_missing)
summary.round(1)

*Your answer here.*

## Task 4 — Market clearing (12 points)

(a) Clear the hour below as a **welfare-maximising LP** in Linopy (M2.1): maximise the value of accepted demand bids minus the cost of accepted supply offers, subject to supply = demand. Write `clear_market(supply, demand)` returning the clearing price (€/MWh, read from the dual — mind the sign), the accepted quantities and the welfare (€).
(b) A coal plant (fuel 14 €/MWh_th, efficiency 0.38, 0.34 t CO₂/MWh_th) competes with a gas plant (fuel 30 €/MWh_th, efficiency 0.55, 0.20 t CO₂/MWh_th). Write `switching_price(...)`: the CO₂ price at which their marginal costs are equal.
(c) In two sentences: who sets the price in (a), and why is that consistent with the accepted and rejected bids?

In [ ]:
SUPPLY = pd.DataFrame({"MW": [300, 350, 400, 500, 300, 200], "EUR_per_MWh": [0, 12, 45, 89, 94, 129]},
                      index=pd.Index(["solar", "nuclear", "hydro", "ccgt", "coal", "ocgt"], name="offer"))
DEMAND = pd.DataFrame({"MW": [1200, 200, 150, 100], "EUR_per_MWh": [3000, 100, 80, 60]},
                      index=pd.Index(["inflexible", "industry_a", "industry_b", "industry_c"], name="bid"))

In [ ]:
def clear_market(supply: pd.DataFrame, demand: pd.DataFrame) -> dict:
    """Clearing price (EUR/MWh, positive), accepted supply and demand (MW) and welfare (EUR) of one hour."""
    ...  # YOUR CODE HERE


def switching_price(fuel_a, eff_a, ef_a, fuel_b, eff_b, ef_b) -> float:
    """CO2 price (EUR/t) at which plants a and b have equal marginal cost (fuel EUR/MWh_th, t CO2/MWh_th)."""
    ...  # YOUR CODE HERE

In [ ]:
result = clear_market(SUPPLY, DEMAND)
p = result["price"]
assert np.isclose(result["supply"].sum(), result["demand"].sum())
assert p > 0, "a clearing price in this market is positive - check the sign of the dual"
assert (SUPPLY.EUR_per_MWh[result["supply"] > 1e-6] <= p + 1e-6).all(), "accepted offers are at or below the price"
assert (DEMAND.EUR_per_MWh[result["demand"] > 1e-6] >= p - 1e-6).all(), "accepted bids are at or above the price"
co2 = switching_price(14, 0.38, 0.34, 30, 0.55, 0.20)
assert np.isclose((14 + co2 * 0.34) / 0.38, (30 + co2 * 0.20) / 0.55)
print(f"price {p:.2f} EUR/MWh, volume {result['demand'].sum():.0f} MW, welfare {result['welfare']:,.0f} EUR; "
      f"switching price {co2:.2f} EUR/t")

*Your answer here.*

## Task 5 — A dispatch LP with a ramp limit (16 points)

Demand is **400, 900 and 500 MW** in three consecutive hours. A base-load plant (800 MW, 20 €/MWh) can change its output by at most **200 MW per hour**; a peaker (500 MW, 100 €/MWh) has no ramp limit.

(a) Write `dispatch()`: the cost-minimising LP in Linopy, returning the cost, both plants' output and the three hourly prices.
(b) Reproduce it in PyPSA (`pypsa_dispatch()`) and show that cost and prices agree to 10⁻⁶.
(c) One hour has a **negative** price although both plants have positive costs. Explain it with the KKT conditions in three or four sentences: which constraint binds, what is its shadow price, and how do the two add up to the price?

In [ ]:
DEMAND_MW = np.array([400.0, 900.0, 500.0])


def dispatch() -> dict:
    """Cost (EUR), output of 'base' and 'peaker' (MW) and hourly prices (EUR/MWh) of the 3-hour LP."""
    ...  # YOUR CODE HERE


def pypsa_dispatch() -> pypsa.Network:
    """The same 3-hour dispatch in PyPSA, optimised."""
    ...  # YOUR CODE HERE

In [ ]:
own = dispatch()
n = pypsa_dispatch()
assert np.allclose(own["base"] + own["peaker"], DEMAND_MW)
assert np.isclose(n.objective, own["cost"], rtol=1e-6)
assert np.abs(n.buses_t.marginal_price.iloc[:, 0].to_numpy() - own["price"]).max() < 1e-6
assert np.all(np.abs(np.diff(own["base"])) <= 200 + 1e-6)
print(f"cost {own['cost']:,.0f} EUR | base {own['base']} | peaker {own['peaker']} | prices {own['price']}")

*Your answer here.*

## Task 6 — A reservoir and its water value (12 points)

A hydro plant with a 100 MW turbine may release **550 MWh** of water over the day `PRICES` below (€/MWh); it is a price-taker and has no other costs.

(a) Write `reservoir()`: the revenue-maximising LP, returning the dispatch, the revenue and the **water value** (the dual of the energy constraint, as a positive €/MWh).
(b) Check the KKT pattern: in every hour the turbine runs at full power the price is at least the water value; where it is off, at most.
(c) In three sentences: why would offering the water at 0 €/MWh in the day-ahead market be a mistake, and what should the plant bid instead?

In [ ]:
PRICES = pd.Series([62, 58, 55, 54, 56, 63, 78, 95, 102, 98, 88, 80, 74, 70, 72, 79, 90, 108, 121, 115, 104, 92, 81, 70],
                   index=pd.RangeIndex(24, name="hour"), dtype=float)


def reservoir() -> dict:
    """Dispatch (MW), revenue (EUR) and water value (EUR/MWh, positive) of the 550 MWh day."""
    ...  # YOUR CODE HERE

In [ ]:
res = reservoir()
g, wv = res["dispatch"], res["water_value"]
assert np.isclose(g.sum(), 550) and wv > 0
assert (PRICES[g > 100 - 1e-6] >= wv - 1e-6).all(), "full power only where the price is at least the water value"
assert (PRICES[g < 1e-6] <= wv + 1e-6).all(), "off only where the price is at most the water value"
print(f"revenue {res['revenue']:,.0f} EUR, water value {wv:.1f} EUR/MWh, running in hours {list(g[g > 1e-6].index)}")

*Your answer here.*

## Task 7 — Code review: a forecasting function with leaks (18 points)

A colleague sends you this day-ahead demand forecast and reports an excellent MAE. The data are the forecasting module's (`load_data`, `temperature_forecast`); the forecast is for the day-ahead market, issued at noon on D−1.

(a) List **three** ways in which `flawed_forecast_mae` uses information it would not have, and for each say what it does to the reported MAE.
(b) Write `honest_forecast_mae()`: a corrected version — split in time (train on 2023–2024, test on 2025), use only features known at noon on D−1, the temperature **forecast**, holidays treated like Sundays — and report its MAE on 2025.
(c) In two sentences: what would you tell your colleague about the reported number?

In [ ]:
DATA = load_data()
DATA["temperature_forecast"] = temperature_forecast(DATA)


def flawed_forecast_mae() -> float:
    """A colleague's day-ahead forecast. Do not change this function."""
    d = DATA
    X = pd.DataFrame({"hour": d.index.hour, "weekday": d.index.dayofweek, "day_of_year": d.index.dayofyear,
                      "temperature": d.temperature, "demand_1h_ago": d.demand.shift(1)}).dropna()
    y = d.demand.loc[X.index]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    model = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(X_train, y_train)
    return mae(model.predict(X_test), y_test)


print(f"reported MAE: {flawed_forecast_mae():.1f} MW")

*Your answer here.*

In [ ]:
def honest_forecast_mae() -> float:
    """MAE (MW) on 2025 of a day-ahead forecast that uses only information available at noon on D-1."""
    ...  # YOUR CODE HERE

In [ ]:
flawed, honest_ = flawed_forecast_mae(), honest_forecast_mae()
assert honest_ > 1.5 * flawed, "an honest day-ahead forecast cannot match a leaky one"
print(f"reported {flawed:.1f} MW, honest {honest_:.1f} MW")

*Your answer here.*

## Task 8 — A newsvendor decision and a brief (14 points)

A supplier buys its day-ahead demand forecast. On its sunny-weekend portfolio a **surplus** is expensive — it must be sold back at deeply negative prices, costing **80 €/MWh** — while a **shortfall** costs only **20 €/MWh**. `ERRORS_2024` are honest past errors (actual − forecast, MW) of its forecast `FORECAST_2025`, with `ACTUAL_2025` the outcome.

(a) Write `cost(schedule, short, long)` and `best_alpha(short, long)`. Which quantile should the supplier buy, and should it buy more or less than the forecast?
(b) Compare the 2025 cost of buying the forecast with buying the forecast plus the α*-quantile of `ERRORS_2024`.
(c) Write a brief of at most 120 words to the supplier's commercial manager: the recommendation, the saving, and one risk of the approach.

In [ ]:
rng = np.random.default_rng(42)
FORECAST_2025 = pd.Series(1000 + 200 * np.sin(np.arange(8760) * 2 * np.pi / 24), name="MW")
ERRORS_2024 = pd.Series(rng.normal(0, 60, 8784))
ACTUAL_2025 = FORECAST_2025 + rng.normal(0, 60, 8760)

In [ ]:
def cost(schedule, short: float, long: float, actual=ACTUAL_2025) -> float:
    """Imbalance cost (EUR) of buying `schedule` when `actual` is consumed."""
    ...  # YOUR CODE HERE


def best_alpha(short: float, long: float) -> float:
    """Cost-minimising quantile to buy."""
    ...  # YOUR CODE HERE

In [ ]:
alpha = best_alpha(20, 80)
schedule = FORECAST_2025 + np.quantile(ERRORS_2024, alpha)
point, better = cost(FORECAST_2025, 20, 80), cost(schedule, 20, 80)
assert better < point, "buying the right quantile must be cheaper here"
assert np.isclose(best_alpha(50, 50), 0.5)
print(f"alpha* = {alpha:.2f}, shift {np.quantile(ERRORS_2024, alpha):+.1f} MW: "
      f"cost {point / 1e6:.3f} -> {better / 1e6:.3f} MEUR ({100 * (1 - better / point):.1f} % saved)")

*Your answer here.*